# X5_Q3 — Hipótesis parámetro↔variable + tabla final

Tercer y último notebook de **X5_alt_v2** (ver
[`docs/plans/X5_alternativo.md`](../docs/plans/X5_alternativo.md) sección 10,
documento rector [`docs/plans/x5_contexto_conversacion.md`](../docs/plans/x5_contexto_conversacion.md)).
TODOs en `docs/tracking/todos.md`, sección X5_alt_v2.

Consume la tabla de estado de `X5_Q1.ipynb`
(`resources/x5_alt/{valor}_tabla_estado_v2.csv`) y el export estructurado de
métricas de `X5_Q2.ipynb` (`resources/x5_alt/{valor}_q2_5_export_metricas.json`)
para traducir la evidencia descriptiva acumulada en hipótesis explícitas
`parámetro X5 ↔ variable` — sin implementarlas todavía como reglas de
trading (eso queda para una etapa posterior de backtesting).

Gate de arranque (`[Q3-1]`): requiere `Q1-EXEC` y `Q2-EXEC` ya satisfechos —
ver `docs/tracking/done.md`.

Mismo input por lista `valores` que `X5_Q1.ipynb`/`X5_Q2.ipynb` (no el
`valor` singular de v1).

In [1]:
valores = ['BTCUSD']  # lista de activos; default solo BTCUSD (ver plan X5_alternativo.md §10.2)


## 0. Preparación

In [2]:
import json
import sys
from pathlib import Path

import pandas as pd

# cwd = scripts/ al correr en Jupyter; fallback a __file__ por si se ejecuta como script
SCRIPTS_DIR = Path.cwd() if (Path.cwd() / 'config.py').exists() else Path(__file__).resolve().parent
sys.path.insert(0, str(SCRIPTS_DIR))
import config as cfg  # noqa: E402

pd.set_option('display.width', 140)  # evita que pandas trunque columnas al imprimir DataFrames anchos

print(f"Activos: {valores}")


Activos: ['BTCUSD']


## 1. Carga de la tabla de estado de Q1 y el export de métricas de Q2 (Q3-1)

Lee, por cada activo de `valores`, `{valor}_tabla_estado_v2.csv` (`X5_Q1.ipynb`,
Sección 10) y `{valor}_q2_5_export_metricas.json` (`X5_Q2.ipynb`, Sección 14) —
ninguno de los dos se recalcula acá. Si falta alguno, corrió incompleto el
notebook correspondiente.

In [3]:
def _cargar_tabla_estado(valor):
    path = cfg.CARPETA_X5_ALT / f'{valor}_tabla_estado_v2.csv'
    if not path.exists():
        raise FileNotFoundError(f"No existe {path} — correr X5_Q1.ipynb primero para generarla.")
    return pd.read_csv(path, parse_dates=['DateTime'])


def _cargar_export_metricas(valor):
    path = cfg.CARPETA_X5_ALT / f'{valor}_q2_5_export_metricas.json'
    if not path.exists():
        raise FileNotFoundError(f"No existe {path} — correr X5_Q2.ipynb primero para generarlo.")
    with open(path, encoding='utf-8') as f:
        return json.load(f)


tablas = {valor: _cargar_tabla_estado(valor) for valor in valores}
export_metricas = {valor: _cargar_export_metricas(valor) for valor in valores}

for valor in valores:
    tabla = tablas[valor]
    print(f"{valor}: tabla_estado {tabla.shape[0]} filas x {tabla.shape[1]} columnas — "
          f"{tabla['DateTime'].min()} → {tabla['DateTime'].max()} | "
          f"export_metricas: {len(export_metricas[valor])} variables")


BTCUSD: tabla_estado 41892 filas x 72 columnas — 2022-01-01 03:00:00 → 2026-10-12 18:00:00 | export_metricas: 63 variables


## 2. Alcance de este notebook (Q3-3)

**`X5_Q3.ipynb` no implementa reglas de trading.** Su único producto son
hipótesis explícitas del tipo `parámetro X5 = f(variable(s) de estado del
activo)` (`N`, `A`, `factor_B`, `K`, `N_EXP`, `LAMBDA`), apoyadas en la
evidencia descriptiva de `X5_Q2.ipynb` y en el pre-registro de
[`X5_alternativo.md`](../docs/plans/X5_alternativo.md) sección 10.5 (`[PRE-1]`).

Quedan explícitamente fuera de este notebook:

- Implementar cualquiera de estas relaciones como código de producción en
  `X0_data_supports.py`/`X1_trading.py`.
- Validar las hipótesis con backtesting — eso es responsabilidad de una
  etapa posterior (puente a `X4_backtester.py` / Fase 3 de `X5_alt`, ver
  TO DO "X5 — estudio posterior de parámetros manipulables y variables
  explicativas" en `docs/tracking/todos.md`).

Una hipótesis con evidencia fuerte en Q2 sigue siendo una hipótesis hasta que
el backtesting la confirme.

## 3. Vista unificada de variables explicativas + precio (Q3-8)

Tabla/export, por activo, con `DateTime`, `Close` y las variables explicativas
**relevantes** — para inspección conjunta (graficar, revisar a ojo), no para
modelar. No recalcula nada: reutiliza el `nivel_evidencia_descriptiva` ya
calculado en `Q2-5`/`Q2-11` dentro de `export_metricas`.

"Relevante" = `nivel_evidencia_descriptiva` en `{alta, media}` contra
`retorno_log_1h` — el target principal. `Close` (nivel) es solo referencial
en toda la síntesis de `Q2-11`: ya se vio ahí que casi ninguna variable
sobrevive el filtro de evidencia contra el nivel de precio, a diferencia del
retorno.

In [4]:
TARGET_PRINCIPAL_Q3 = 'retorno_log_1h'  # Close es solo referencial (ver síntesis Q2-11)
NIVELES_RELEVANTES_Q3_8 = {'alta', 'media'}


def _variables_relevantes(valor):
    metricas = export_metricas[valor]
    return sorted(
        variable for variable, info in metricas.items()
        if info['targets'][TARGET_PRINCIPAL_Q3]['nivel_evidencia_descriptiva'] in NIVELES_RELEVANTES_Q3_8
    )


vistas_unificadas = {}
for valor in valores:
    variables_relevantes = _variables_relevantes(valor)
    vista = tablas[valor][['DateTime', 'Close'] + variables_relevantes].copy()
    vistas_unificadas[valor] = vista
    out_path = cfg.CARPETA_X5_ALT / f'{valor}_q3_8_vista_unificada.csv'
    vista.to_csv(out_path, index=False)
    print(f"{valor}: {len(variables_relevantes)} variables relevantes (nivel {NIVELES_RELEVANTES_Q3_8} "
          f"contra {TARGET_PRINCIPAL_Q3}) -> {out_path.name}")


BTCUSD: 19 variables relevantes (nivel {'alta', 'media'} contra retorno_log_1h) -> BTCUSD_q3_8_vista_unificada.csv


## 4. Hipótesis prioritaria: N = f(drawdown_activo) (Q3-2)

**Qué controla**: `N` (`n_sizes_ejecucion`) es la cantidad de soportes activos
que X1 mantiene por activo. ↑N → más cobertura del rango y entradas más
finas, pero capital más fragmentado. ↓N → posiciones más concentradas,
cobertura más gruesa.

**Hipótesis** (documento rector [`x5_contexto_conversacion.md`](../docs/plans/x5_contexto_conversacion.md)
sección 6, pre-registrada en [`X5_alternativo.md`](../docs/plans/X5_alternativo.md)
§10.5 `[PRE-1]`): drawdown del activo sano (poco profundo/reciente) + volatilidad
baja/normal → `N` alto. Drawdown profundo/no sano + volatilidad alta → `N` bajo.
Nunca drawdown/equity de la cuenta — solo del precio del activo.

**Signo esperado**: relación directa entre la "salud" del drawdown (menos
profundo, más cercano a 0) y `N` — cuanto más profundo/negativo el drawdown,
más bajo `N`.

**Variables candidatas**: profundidad y duración del drawdown en distintas
ventanas (`drawdown_7d`/`30d`/`90d`/`365d`, `drawdown_historico`), tiempo
desde el máximo, velocidad de caída/recuperación, volatilidad corta/larga.

**Evidencia descriptiva disponible** (`Q2-11`, contra `retorno_log_1h` en
BTCUSD): las ventanas cortas de drawdown tienen evidencia **alta** y signo
consistente con la hipótesis — `x3_drawdown_20` (Pearson 0.28), `x3_drawdown_50`
(0.19) y `drawdown_7d` (0.11) correlacionan positivo con el retorno siguiente
(drawdown menos profundo → retorno algo mayor), estables en el tiempo y sin
dependencia de régimen. Las ventanas largas (`drawdown_90d`/`365d`/`historico`)
caen a evidencia **baja** — la señal se debilita cuanto más se alarga la
ventana. Esto es evidencia sobre la relación variable↔retorno futuro, no
sobre el parámetro `N` en sí — no reemplaza el backtesting.

**Rango permitido**: mismo rango de exploración ya usado por X5
(`X5_PARAM_RANGES`, `config.py`, ver [`pars_configurables.md`](../docs/context/pars_configurables.md)) —
`(50, 200)` para BTCUSD/ETHUSD, `(40, 180)` para TSLA/GOOGL/NVDA/AMZN.

**Riesgo económico asociado**: `N` bajo en drawdown profundo reduce la
exposición simultánea (menos soportes = menos entradas abiertas a la vez),
acotando la pérdida potencial agregada por activo, a costa de menor cobertura
si el drawdown resulta ser temporal. `N` alto en drawdown sano maximiza
cobertura/entradas finas, pero fragmenta capital y multiplica las posiciones
simultáneas si el drawdown se profundiza inesperadamente después.

**Cómo validar**: backtesting (`X4_backtester.py`) comparando desempeño con
`N` dinámico `N=f(drawdown_activo)` contra el `N` estático actual (180 fijo),
por activo — el análisis descriptivo de Q2 es insumo para la hipótesis, no
su validación.

## 5. Hipótesis de `A` y `factor_B`: ambos `= f(volatilidad[, tendencia])` (Q3-5)

**Qué controlan**: `A` (USD) cumple dos roles en `X1_trading.py` — filtro de
creación de Órdenes en Espera (`(P0-Pi)*L >= A`) y umbral de ganancia para
activar el primer Stop Loss (SL) ganador del trailing
(`ganancia = (P0-Pi)*L >= A`). `factor_B` es la proporción de `A` que se usa
como distancia del trailing una vez activo (`B = factor_B * A`): factor bajo
→ trailing ajustado (asegura ganancia rápido); factor alto → trailing
holgado (tolera más ruido, deja correr la tendencia). Ver
[`pars_configurables.md`](../docs/context/pars_configurables.md).

**Hipótesis** (documento rector [`x5_contexto_conversacion.md`](../docs/plans/x5_contexto_conversacion.md)
secciones 7-8, pre-registradas en [`X5_alternativo.md`](../docs/plans/X5_alternativo.md)
§10.5 `[PRE-1]`):
- `A = f(volatilidad)`: volatilidad alta → `A` más alto (OE más espaciadas,
  trailing se activa más tarde). Volatilidad baja → `A` más bajo.
- `factor_B = f(volatilidad, tendencia)`: volatilidad alta → `factor_B` más
  alto (evita cierre por ruido). Volatilidad baja → `factor_B` más bajo
  (asegura ganancia rápido). Tendencia fuerte y persistente → `factor_B`
  más alto. Mercado lateral/ruidoso → `factor_B` bajo/intermedio — relación
  explícitamente "por comprobar con backtesting", no confirmada.

**Dualidad de roles de `A`**: el filtro de OE y la activación del trailing
tiran en direcciones distintas frente a un mismo contexto — con drawdown
profundo, un `A` alto reduce entradas nuevas pero también retrasa la
protección de una posición ganadora. Por eso el efecto neto del drawdown
sobre `A` no tiene dirección única; podría requerir separar
`A_OE`/`A_TRAILING` más adelante (decisión pendiente, documento rector
sección 21). Por ahora se mantiene un único `A`.

**Variables candidatas**: `x3_atr_pct` (ATR_14/Close, ya relativo al
precio — base natural para `A = multiplicador_ATR * x3_atr_pct * Close * L`,
fórmula sugerida en el documento rector) y `x3_bb_width` (ancho de Bandas de
Bollinger normalizado) como las dos medidas de volatilidad no redundantes
entre sí (Pearson 0.74, bajo el umbral de redundancia de `Q2-12`); `adx_14`
(fuerza de tendencia sin dirección, independiente de ambas: Pearson
0.28-0.49) para el componente de tendencia de `factor_B`;
`regimen_tendencia` (bull/bear/sideways, construida desde
`adx_14`+`x3_trend_slope_50`) para distinguir lateral/tendencial.
`x3_vol_24h` queda descartada como candidata adicional: redundante con
`x3_atr_pct` (Pearson 0.91, `Q2-12`).

**Evidencia descriptiva disponible y su límite**: `A`/`factor_B` no son
hipótesis de predicción de retorno sino de escala de la mecánica de órdenes
frente al estado del activo — por eso no sorprende que `x3_atr_pct`,
`x3_bb_width` y `adx_14` tengan `nivel_evidencia_descriptiva` **baja**
contra `retorno_log_1h` en `Q2-11` (\|Pearson\| < 0.02 los tres): no se
espera que predigan dirección, sino que describan la escala de movimiento
vigente. El propio documento rector lo anticipa (sección 17): "no basta con
correlacionar A o factor_B con una variable — la relación debe evaluarse
mediante simulación". Dato que sí aporta valor descriptivo (`Q2-14`): varias
de estas variables de volatilidad (`x3_vol_24h`, `x3_atr_pct`, `x3_bb_width`,
además de `tick_volume_z`/`spread_relativo`/`volumen_relativo`) muestran
correlación de distancia 0.17-0.34 contra `retorno_log_1h`, muy por encima
de su Pearson casi nulo — dependencia no lineal real, aunque sin definir
signo ni servir de validación.

**Rango permitido**: `A ∈ (2.0, 20.0)` USD, `factor_B ∈ (0.2, 0.7)`
(`config.py`, `pars_configurables.md`) — el `B` productivo histórico
(1.5 USD) con `A=3` implica `factor_B≈0.5`, dentro de ese rango.

**Riesgo económico asociado**: `A` alto reduce la densidad de entradas
nuevas pero retrasa la protección de ganancias ya abiertas; `A` bajo
protege antes pero puede sacar de la posición con ganancias mínimas por
ruido. `factor_B` alto tolera más ruido antes de cerrar pero devuelve más
ganancia flotante si el precio revierte; `factor_B` bajo asegura ganancia
más rápido pero puede cerrar por una corrección menor antes de que la
tendencia continúe.

**Cómo validar**: backtesting (`X4_backtester.py`), nunca el análisis
descriptivo de Q2 — el documento rector (sección 17) exige simular
`estado X_t → configuración θ_t → ejecución X1 → resultado`, midiendo por
separado, para `A`: densidad/distancia de OE nuevas vs. retraso del primer
trailing; y para `factor_B`: frecuencia de stop por ruido, ganancia
protegida vs. devuelta, y capacidad de capturar tendencias largas — por
régimen.

## 6. Hipótesis de `K`, `N_EXP` y `LAMBDA` (Q3-6)

**Qué controlan** (`X0_data_supports.py`, ver
[`pars_configurables.md`](../docs/context/pars_configurables.md)): `K`
pondera aislamiento futuro vs. pasado en
`y = Low_left + High_left + K*(Low_right+High_right)`; `N_EXP` controla la
recencia en `w = t^N_EXP`; `LAMBDA` penaliza dispersión desigual en
`FO = mean(z) - LAMBDA*cv(H_n)`. Los tres actúan dentro del algoritmo de
soportes de X0, no directamente en X1.

**Por qué requieren dos niveles de análisis** (documento rector sección 18)
— ninguno de los dos lo mide el análisis descriptivo de `X5_Q2.ipynb` (que
corre sobre precio/variables de estado, no sobre soportes ni órdenes); Q2
solo aporta evidencia indirecta sobre el estado del activo que podría
explicar por qué convendría variar cada parámetro:
1. **Nivel 1 — efecto sobre los soportes (X0)**: cantidad, calidad
   individual, recencia, cobertura del rango, concentración, distancia
   entre niveles, simetría de aislamiento, estabilidad.
2. **Nivel 2 — efecto posterior en X1**: órdenes creadas/activadas,
   cierres, dinero cerrado, MAE/MFE, duración, estabilidad por período.
   "No debe concluirse que un valor es bueno solo porque genera soportes
   visualmente bien distribuidos" — debe demostrar utilidad operativa en
   X1.

**`K`** — candidata, sin función explícita todavía (documento rector
sección 12; conviene tratarlo relativamente estable por activo, no
dinámico vela a vela). Hipótesis: mercado estable y niveles persistentes →
`K` alto; muchos cambios de régimen/rupturas → `K` bajo. Variables
candidatas: `duracion_en_rango` (períodos desde la última ruptura de
rango — proxy directo de persistencia de niveles) y `ruptura_rango`
(±1/0 estilo Donchian-20 — frecuencia de rupturas). No existe hoy en el
catálogo una variable de "tasa de rechazo posterior" o "simetría
aislamiento previo/posterior" propiamente dicha — construirla exigiría
volver a los propios `Low_left/right`/`High_left/right` de
`calcular_distancias` en X0, fuera del alcance de Q3.

**`N_EXP`** — "buen candidato a depender dinámicamente del estado del
activo" (documento rector sección 13). Hipótesis: mercado
estable/repetitivo → `N_EXP` bajo; cambio de régimen o volatilidad
creciente → `N_EXP` alto. Variable candidata principal: `cambio_volatilidad`
(`x3_vol_24h / x3_vol_7d`) — es literalmente el "ratio de volatilidad
corta/larga" que pide el documento rector; marcada
`depende_de_algun_regimen=True` (drawdown, tendencia y volatilidad) en
`Q2-11`, consistente con que por construcción capta inestabilidad de
contexto, aunque su Pearson directo contra `retorno_log_1h` sea bajo
(0.003, evidencia "baja" — no se espera que prediga retorno).
`frecuencia_extremos_30d` (fracción de velas con retorno atípico en 30d)
como candidata secundaria para cambio de régimen.

**`LAMBDA`** — hipótesis: mercado lateral/rango amplio con soportes
concentrados → `LAMBDA` alto; tendencia fuerte con evidencia concentrada →
`LAMBDA` bajo. Debe analizarse junto con `N` (la penalización por
dispersión cambia según cuántos soportes hay). Variables candidatas:
`x3_bb_width` (amplitud relativa del rango reciente) y las mismas
`ruptura_rango`/`duracion_en_rango` de `K`, aquí interpretadas como
lateral-vs-tendencial en vez de persistencia puntual. **Inconsistencia sin
resolver** (documento rector sección 14.1): el valor productivo de
`LAMBDA` (`0.2`) no está cubierto por el rango de exploración de X5
documentado (`0.001`-`0.02`) — no deben sacarse conclusiones de una
exploración que no contiene el valor productivo de referencia; queda
pendiente decidir si el rango está desactualizado, fuera del alcance de Q3.

**Cómo validar**: backtesting en dos niveles sobre `X4_backtester.py`
(nivel 1 sobre los soportes generados, nivel 2 sobre el resultado de X1) —
nunca solo con el análisis descriptivo de Q2, por las mismas razones que
`A`/`factor_B` en `Q3-5`.

## 7. Vector `X → parámetro Y` consolidado por parámetro (Q3-7)

Condensa `Q3-2`/`Q3-5`/`Q3-6` a pocas variables explicativas por
parámetro — evitando que todos los parámetros dependan de todas las
variables — como insumo directo para la tabla final de `Q3-4`.
`LOTAJES_M`/`PERDIDA_MAX` quedan fuera (no prioritario / pausado, ver
[`pars_configurables.md`](../docs/context/pars_configurables.md) y
documento rector secciones 10-11).

| Parámetro | Variables candidatas (pocas) | Relación hipotética |
|---|---|---|
| `N` | `x3_drawdown_20`, `x3_drawdown_50`, `drawdown_7d` | drawdown sano (menos profundo) → `N` alto; drawdown profundo → `N` bajo |
| `A` | `x3_atr_pct`, `x3_bb_width` | volatilidad alta → `A` alto; volatilidad baja → `A` bajo (dirección neta sobre drawdown ambigua, ver `Q3-5`) |
| `factor_B` | `x3_atr_pct`, `adx_14` / `regimen_tendencia` | volatilidad alta o tendencia fuerte → `factor_B` alto; volatilidad baja → `factor_B` bajo |
| `K` | `duracion_en_rango`, `ruptura_rango` | niveles persistentes/mercado estable → `K` alto |
| `N_EXP` | `cambio_volatilidad`, `frecuencia_extremos_30d` | cambio de régimen o volatilidad creciente → `N_EXP` alto |
| `LAMBDA` | `x3_bb_width`, `duracion_en_rango` (junto con `N`) | rango amplio/lateral con soportes concentrados → `LAMBDA` alto |
| `LOTAJES_M` | — | sin hipótesis dinámica — fijo (rango degenerado `(1,1)`) |
| `PERDIDA_MAX` | — | pausado — en evaluación para una capa externa de gestión de riesgo |

**Criterio de selección**: se priorizó, por parámetro, la variable con
mejor evidencia descriptiva disponible (`Q2-11`/`Q2-14`) que no fuera
redundante con otra ya elegida para el mismo parámetro (`Q2-12`) — ej.
`x3_vol_24h` se descarta en `A` por ser redundante con `x3_atr_pct`
(Pearson 0.91); `x3_atr_14` (sin normalizar) se descarta frente a
`x3_atr_pct` por estar ya expresado en unidades relativas al precio.
Ninguna de estas tuplas está validada — son el insumo de entrada a `Q3-4`,
que agrega evidencia y prioridad de prueba.